# Demanda eléctrica por área del SEIN: comprensión y preparación de los datos

**TB1 · Metodología para Data Science (1ACC0240) · UPC · ciclo 2026-2 · Grupo 3**

Notebook 01 de CRISP-DM para el caso peruano: comprensión de los datos y preparación. El problema, la regla de
información y el plan están en la
[spec del proyecto](https://github.com/R0SEWT/tetouan-power-forecast/blob/develop/docs/superpowers/specs/2026-10-23-demanda-sein-areas-design.md);
las fuentes y su calidad, en
[`docs/peru/`](https://github.com/R0SEWT/tetouan-power-forecast/tree/develop/docs/peru).

Lee las capas que arman `00_medallon_coes` y `00c_exogenas`: la demanda del SEIN y de sus tres áreas cada 30
minutos (IEOD del COES, de 2016 a hoy) y las exógenas (ERA5, ICEN, Niño 1+2, SAIDI y eventos del COES).

# Configuración

In [ ]:
import os
import sys
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB and "TETOUAN_DATA" not in os.environ:
    RAMA = "feature/nucleo-sein"  # cambiar a "develop" cuando se fusione
    !pip install -q "git+https://github.com/R0SEWT/tetouan-power-forecast@{RAMA}"
    from tetouan_power_forecast.medallon import usar_drive
    usar_drive()
elif not EN_COLAB:
    os.environ.setdefault("TETOUAN_DATA", str(Path.cwd().parent / "data"))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from tetouan_power_forecast.coes import cargar_demanda_areas
from tetouan_power_forecast.evaluacion import errores
from tetouan_power_forecast.exogenas import fallas_en_curso
from tetouan_power_forecast.medallon import raiz_datos
from tetouan_power_forecast.variables_sein import (
    AREAS,
    HORIZONTES,
    REFERENCIAS,
    columnas_variante,
    dataset_area,
    filas_comunes,
    particion,
)

RAIZ = raiz_datos()
COLORES = {"NORTE": "#E69F00", "CENTRO": "#0072B2", "SUR": "#009E73", "SEIN": "#555555"}
pd.set_option("display.width", 160)

demanda = cargar_demanda_areas()
silver_exo = RAIZ / "exogenas" / "silver"
temperatura = pd.read_parquet(silver_exo / "temperatura_areas.parquet")
icen = pd.read_parquet(silver_exo / "icen.parquet")
nino12 = pd.read_parquet(silver_exo / "nino12_semanal.parquet")
saidi = pd.read_parquet(silver_exo / "saidi_saifi.parquet")
eventos = pd.read_parquet(silver_exo / "eventos_coes.parquet")
try:
    import holidays
    FERIADOS = list(holidays.PE(years=range(2016, 2027)))
except ImportError:
    FERIADOS = []
print(f"Demanda: {len(demanda):,} periodos de 30 min, de {demanda['DateTime'].min()} a {demanda['DateTime'].max()}")
print(f"Temperatura ERA5: {len(temperatura):,} horas · ICEN: {len(icen)} meses · Niño 1+2: {len(nino12)} semanas")
print(f"SAIDI: {len(saidi):,} filas · eventos del COES: {len(eventos):,} · feriados: {len(FERIADOS)}")

# Comprensión de los datos

## Fuente, procedencia y auditoría

La demanda sale de la hoja `DEMANDA_AREAS` del anexo diario del IEOD del COES. Cada valor es la demanda media de un
periodo de 30 minutos, en MW, con la marca de tiempo al **final** del periodo (00:30 es el periodo de 00:00 a
00:30). En el IEOD, Norte + Centro + Sur = SEIN. La auditoría del silver:

In [ ]:
display(Markdown((RAIZ / "coes" / "silver" / "auditoria.md").read_text()))

In [ ]:
anio = demanda["DateTime"].dt.year
calidad = pd.DataFrame({
    "periodos": demanda.groupby(anio).size(),
    **{f"faltan {a.lower()}": demanda[a].isna().groupby(anio).sum() for a in AREAS},
    "N+C+S−SEIN máx (MW)": (demanda[AREAS].sum(axis=1, min_count=3) - demanda["SEIN"]).abs().groupby(anio).max(),
    "no positivos": (demanda[AREAS] <= 0).any(axis=1).groupby(anio).sum(),
})
display(calidad.round(2))

## Escala y crecimiento de cada área

In [ ]:
mensual = demanda.set_index("DateTime")[["SEIN", *AREAS]].resample("MS").mean()
fig, eje = plt.subplots(figsize=(14, 4.5))
for c in ["SEIN", *AREAS]:
    eje.plot(mensual.index, mensual[c], color=COLORES[c], label=c.capitalize())
eje.set_ylabel("Demanda media mensual (MW)")
eje.legend(ncol=4)
eje.set_title("Demanda media mensual del SEIN y de sus áreas")
plt.tight_layout()
plt.show()
anual = demanda.groupby(anio)[["SEIN", *AREAS]].mean()
display(anual.round(0))
print("Participación de cada área en el SEIN (%):")
display((100 * anual[AREAS].div(anual["SEIN"], axis=0)).round(1))

## Perfil diario y deriva del pico

La investigación encontró que el pico diario del SEIN se fue del atardecer al mediodía. Se mide por área: la
fracción de días cuyo máximo cae entre las 18 y las 23 h.

In [ ]:
inicio = demanda["DateTime"] - pd.Timedelta(minutes=30)  # el día y la hora se asignan por el inicio del periodo
diaria = demanda.assign(dia=inicio.dt.normalize(), hora=inicio.dt.hour)
filas = []
for a in ["SEIN", *AREAS]:
    idx = diaria.dropna(subset=[a]).groupby("dia")[a].idxmax()
    horas = diaria.loc[idx, ["dia", "hora"]]
    filas.append(horas.assign(area=a, anio=horas["dia"].dt.year, punta=horas["hora"].between(18, 22)))
picos = pd.concat(filas)
deriva = picos.pivot_table(index="anio", columns="area", values="punta", aggfunc="mean") * 100
display(deriva.round(1).rename_axis(columns="% de días con el máximo entre 18 y 23 h"))

fig, ejes = plt.subplots(1, 3, figsize=(16, 4), sharey=False)
for eje, a in zip(ejes, AREAS):
    perfil = diaria[diaria["dia"].dt.year.isin([2016, 2021, 2025])].groupby([diaria["dia"].dt.year, "hora"])[a].mean()
    for anio_, serie in perfil.groupby(level=0):
        eje.plot(serie.index.get_level_values(1), serie / serie.mean(), label=str(anio_))
    eje.set_title(a.capitalize())
    eje.set_xlabel("Hora (inicio del periodo)")
ejes[0].set_ylabel("Demanda / media del día")
ejes[0].legend()
plt.suptitle("Perfil diario medio en tres años: cómo se movió el pico")
plt.tight_layout()
plt.show()

## Semana y feriados

In [ ]:
por_dia = demanda.set_index("DateTime")[AREAS].resample("D").mean().dropna()
semana = por_dia.groupby(por_dia.index.day_of_week).mean() / por_dia.mean()
semana.index = ["lun", "mar", "mié", "jue", "vie", "sáb", "dom"]
display(semana.round(3).T.rename_axis(columns="Consumo del día / promedio"))
if FERIADOS:
    feriado = por_dia.index.isin(pd.to_datetime(FERIADOS))
    habiles = por_dia[(por_dia.index.day_of_week < 5) & ~feriado].mean()
    efecto = 100 * (por_dia[feriado & (por_dia.index.day_of_week < 5)].mean() / habiles - 1)
    print("Feriados entre semana frente a un día hábil (%):", efecto.round(1).to_dict())

## Temperatura y demanda

La temperatura de cada área es la media de sus ciudades en ERA5 (Norte: Piura, Chiclayo y Trujillo; Centro: Lima;
Sur: Arequipa y Tacna). Se compara la temperatura media diaria con la demanda media diaria de los días hábiles.

In [ ]:
temp_dia = temperatura.set_index("fecha_hora").resample("D").mean()
temp_dia.columns = [c.upper() for c in temp_dia.columns]
habiles = por_dia[por_dia.index.day_of_week < 5]
fig, ejes = plt.subplots(1, 3, figsize=(16, 4.5))
correlaciones = {}
for eje, a in zip(ejes, AREAS):
    unido = pd.concat([temp_dia[a], habiles[a] / habiles[a].rolling(365, min_periods=60, center=True).mean()],
                      axis=1, keys=["temp", "demanda_rel"]).dropna()
    correlaciones[a] = unido.corr().iloc[0, 1]
    eje.scatter(unido["temp"], unido["demanda_rel"], s=3, alpha=0.3, color=COLORES[a])
    eje.set(title=f"{a.capitalize()} · r = {correlaciones[a]:.2f}", xlabel="Temperatura media del día (°C)")
ejes[0].set_ylabel("Demanda del día / media móvil anual")
plt.suptitle("Temperatura y demanda en días hábiles (sin la tendencia)")
plt.tight_layout()
plt.show()

## El Niño: 2017, 2023 y 2026

El ICEN mide El Niño Costero (Niño 1+2). Se compara con la demanda de cada mes frente al mismo mes del año anterior,
que quita la estacionalidad pero no el crecimiento (unos +2 a +4 % al año).

In [ ]:
icen_m = icen.set_index("mes")["icen"]
variacion = 100 * (mensual[AREAS] / mensual[AREAS].shift(12) - 1)
fig, (eje1, eje2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
eje1.bar(icen_m.loc["2016":].index, icen_m.loc["2016":], width=25,
         color=np.where(icen_m.loc["2016":] > 0.4, "#D55E00", "#56B4E9"))
eje1.axhline(0.4, color="grey", lw=0.8, ls="--")
eje1.set_ylabel("ICEN")
eje1.set_title("El Niño Costero (ICEN) y la demanda frente al mismo mes del año anterior")
for a in AREAS:
    eje2.plot(variacion.index, variacion[a], color=COLORES[a], label=a.capitalize())
eje2.axhline(0, color="black", lw=0.8)
eje2.set_ylabel("Variación interanual (%)")
eje2.legend(ncol=3)
plt.tight_layout()
plt.show()
for nombre, desde, hasta in [("Niño Costero 2017", "2017-01", "2017-04"), ("Niño Costero 2023", "2023-03", "2023-07"),
                             ("Niño extraordinario 2026", "2026-03", "2026-10")]:
    tabla = pd.concat([icen_m.loc[desde:hasta].rename("ICEN"), variacion.loc[desde:hasta]], axis=1)
    display(tabla.round(2).style.set_caption(nombre))

## Fallas del COES y SAIDI de Osinergmin

Las fallas cuentan como en curso desde su inicio hasta su final, con un máximo de 2 días (los registros de meses
o años son indisponibilidades). El SAIDI es la duración media de los cortes por usuario de cada sistema eléctrico.

In [ ]:
en_curso = pd.Series(fallas_en_curso(demanda["DateTime"], eventos).to_numpy(), index=demanda["DateTime"])
print("Fallas en curso por periodo, por año:")
display(en_curso.groupby(en_curso.index.year).agg(["mean", "max"]).round(2).T)
eventos_mes = eventos.set_index("inicio").resample("MS").size()
norte = saidi[saidi["nombre_empresa"].str.contains("ENOSA|ELECTRONORTE|HIDRANDINA|ELECTRONOROESTE", case=False,
                                                      na=False)]
saidi_norte = norte.groupby("mes")["saidi"].median()
fig, (eje1, eje2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
eje1.plot(eventos_mes.loc["2016":].index, eventos_mes.loc["2016":], color="#555555")
eje1.set_ylabel("Eventos por mes (COES)")
eje2.plot(saidi_norte.loc["2016":].index, saidi_norte.loc["2016":], color=COLORES["NORTE"])
eje2.set_ylabel("SAIDI mensual, mediana\nde los sistemas del norte")
plt.tight_layout()
plt.show()

## La cuarentena de 2020

In [ ]:
semanal = demanda.set_index("DateTime")[AREAS].resample("W").mean()
s20, s19 = semanal.loc["2020-02":"2020-09"], semanal.loc["2019-02":"2019-09"]
s19.index = s19.index + pd.DateOffset(years=1)
comparada = 100 * (s20 / s19.reindex(s20.index, method="nearest") - 1)
fig, eje = plt.subplots(figsize=(12, 4))
for a in AREAS:
    eje.plot(comparada.index, comparada[a], color=COLORES[a], label=a.capitalize())
eje.axvline(pd.Timestamp("2020-03-16"), color="black", lw=0.8, ls="--")
eje.set_ylabel("Frente a la misma semana de 2019 (%)")
eje.legend()
eje.set_title("La cuarentena de 2020 por área")
plt.tight_layout()
plt.show()

## Qué tan difícil es cada área: las referencias ingenuas

MAPE (%) de cada referencia por área, horizonte y año: persistencia, ayer y semana pasada a la misma hora. A 24 h,
"ayer" y la persistencia coinciden.

In [ ]:
filas = []
for nombre_h, h in HORIZONTES.items():
    for a in AREAS:
        d = dataset_area(demanda, a, horizonte=h)
        for anio_, g in d.dropna(subset=["real", *REFERENCIAS]).groupby(d["DateTime"].dt.year):
            fila = {"horizonte": nombre_h, "area": a, "anio": anio_}
            fila.update({ref: errores(g["real"], g[ref])["MAPE (%)"] for ref in REFERENCIAS})
            filas.append(fila)
referencias = pd.DataFrame(filas)
referencias["mejor"] = referencias[REFERENCIAS].idxmin(axis=1)
display(referencias.pivot_table(index="anio", columns=["horizonte", "area"],
                                values=["persistencia", "ayer", "semana_pasada"]).round(2))

# Preparación de los datos

El dataset de cada área y horizonte sale de `dataset_area` (`src/tetouan_power_forecast/variables_sein.py`), con la
regla de información de la spec (§6): la temperatura en t es la última hora publicada; a 24 h, la de t+H es un
pronóstico simulado con un error por día de emisión; el ICEN, el Niño 1+2 y el SAIDI entran según su publicación;
las fallas cuentan como mucho dos días. La partición: entrenamiento 2016-2024, validación 2025 y prueba 2026, con un
hueco de H pasos antes de cada corte.

In [ ]:
filas = []
for nombre_h, h in HORIZONTES.items():
    for a in AREAS:
        d = dataset_area(demanda, a, horizonte=h, temperatura=temperatura, icen=icen, nino12=nino12,
                         eventos=eventos, feriados=FERIADOS)
        d["particion"] = particion(d["DateTime"], h)
        comunes = filas_comunes(d, ["M0", "M1", "M2", "M3"])
        conteo = comunes["particion"].value_counts()
        filas.append({"horizonte": nombre_h, "area": a, "variables_M0": len(columnas_variante(d, "M0")) - 1,
                      "variables_M3": len(columnas_variante(d, "M3")) - 1, **conteo.to_dict(),
                      "desde": comunes["DateTime"].min(), "hasta": comunes["DateTime"].max()})
resumen = pd.DataFrame(filas)
display(resumen)
print(f"Variables de M3 ({a.capitalize()}, {nombre_h}):", columnas_variante(d, "M3"))

In [ ]:
# Chequeos de la regla de información en el dataset real
d = dataset_area(demanda, "CENTRO", horizonte=2, temperatura=temperatura)
horaria = temperatura.set_index("fecha_hora")["centro"]
muestra = d.dropna(subset=["temp"]).sample(200, random_state=0)
esperada = horaria.reindex(pd.DatetimeIndex(muestra["DateTime"]), method="ffill").to_numpy()
assert np.allclose(muestra["temp"], esperada), "la temperatura en t debe ser la última hora publicada"
assert (d["DateTime"].diff().dropna() == pd.Timedelta(minutes=30)).all(), "un periodo cada 30 minutos"
print("Regla de información verificada en una muestra de 200 periodos.")

# Hallazgos

(Se completan con los resultados de la corrida.)